# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_10 — Smart Order Router Simulator

### Research question

How can a smart order router decide where to send child orders across venues when venues differ in liquidity, latency, fees, rebates, fill probability, adverse selection, and market-impact risk?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
06_08_avellaneda_stoikov_market_making
06_09_limit_order_fill_probability_model
06_10_smart_order_router_simulator
```

The previous notebook estimated passive limit-order fill probabilities. This notebook uses those ideas to decide **where** to route orders.

It covers:

1. multi-venue market structure;
2. venue metadata;
3. maker/taker fees and rebates;
4. venue latency;
5. queue and depth proxies;
6. lit and dark venues;
7. fill-probability models by venue;
8. adverse-selection estimates;
9. marketable routing;
10. passive routing;
11. child-order splitting;
12. expected-cost routing;
13. fill-probability-aware routing;
14. fee-minimising routing;
15. latency-minimising routing;
16. liquidity-seeking routing;
17. dark-pool midpoint routing;
18. venue allocation optimisation;
19. parent-order simulation;
20. execution-quality metrics;
21. venue-level TCA;
22. routing policy comparison;
23. stress scenarios;
24. governance flags;
25. audit checks;
26. saved outputs and manifest.

The key idea:

> A smart order router is not just looking for the best price; it is balancing price, fill probability, latency, fees, adverse selection, and liquidity risk.

## 1. Why smart order routing matters

Suppose a strategy wants to buy $Q$ shares.

It can route to:

- a fast lit venue with high taker fees;
- a slow venue with deep liquidity;
- a maker-rebate venue with lower fill probability;
- a dark midpoint venue with uncertain fills;
- a venue with adverse selection risk.

The best venue depends on the objective.

A router can minimise:

$$
\begin{aligned}
ExpectedCost &= SpreadCost \\
&\quad + Fees \\
&\quad + Impact \\
&\quad + AdverseSelection \\
&\quad + OpportunityCost
\end{aligned}
$$

subject to:

- order size;
- time-in-force;
- venue eligibility;
- minimum fill sizes;
- latency;
- risk controls.

The router is the bridge between execution theory and actual child-order placement.

## 2. Venue types

This notebook simulates four venue types:

| Type | Description |
|---|---|
| Lit primary | Deep, fast, visible liquidity |
| Lit maker-rebate | Strong rebates but possible adverse selection |
| Lit low-fee | Cheap taker execution but lower depth |
| Dark midpoint | Midpoint fills, uncertain probability, no displayed quote |

This is simplified, but it captures the real routing trade-off:

> visible certainty versus hidden improvement.

## 3. Routing policies

We compare:

### 1. Primary-only

Route everything to primary venue.

### 2. Cheapest taker

Choose lowest all-in marketable cost.

### 3. Fastest venue

Choose lowest latency.

### 4. Liquidity weighted

Split by visible depth.

### 5. Expected-cost router

Minimise expected cost after fees, impact, and adverse selection.

### 6. Fill-probability-aware router

Use passive fill probability and fallback market order.

### 7. Dark-first midpoint router

Try midpoint dark fills first, then sweep lit venues.

A serious router needs all these diagnostics.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class SmartOrderRouterConfig:
    n_events: int = 60_000
    n_parent_orders: int = 2_500
    seed: int = 42
    initial_mid: float = 100.0
    tick_size: float = 0.01
    parent_order_median_size: float = 7_500.0
    child_order_size: float = 1_000.0
    max_child_orders_per_parent: int = 12
    time_in_force_events: int = 30
    dark_first_wait_events: int = 8
    adverse_selection_horizon: int = 15
    opportunity_cost_bps: float = 1.25
    fallback_cross_after_dark: bool = True
    stress_start_frac: float = 0.65
    stress_end_frac: float = 0.82
    benchmark_repeats: int = 3
    output_subdir: str = "smart_order_router_simulator_v1"

config = SmartOrderRouterConfig()
config

## 5. Venue metadata

Venue metadata includes:

- venue type;
- taker fee;
- maker rebate;
- latency;
- depth multiplier;
- fill intensity;
- adverse-selection multiplier;
- midpoint eligibility;
- minimum fill size.

This is the router’s static configuration.

In [ ]:
def make_venue_metadata():
    venues = pd.DataFrame([
        {
            "venue": "PRIMARY",
            "venue_type": "lit_primary",
            "taker_fee_bps": 0.45,
            "maker_rebate_bps": -0.05,
            "latency_events": 1,
            "depth_multiplier": 1.00,
            "fill_intensity_multiplier": 1.00,
            "adverse_selection_multiplier": 1.00,
            "midpoint": False,
            "min_fill_size": 1.0,
            "eligible_market": True,
            "eligible_limit": True,
        },
        {
            "venue": "REBATE",
            "venue_type": "lit_rebate",
            "taker_fee_bps": 0.65,
            "maker_rebate_bps": -0.28,
            "latency_events": 3,
            "depth_multiplier": 0.75,
            "fill_intensity_multiplier": 0.78,
            "adverse_selection_multiplier": 1.35,
            "midpoint": False,
            "min_fill_size": 1.0,
            "eligible_market": True,
            "eligible_limit": True,
        },
        {
            "venue": "LOWFEE",
            "venue_type": "lit_low_fee",
            "taker_fee_bps": 0.15,
            "maker_rebate_bps": 0.00,
            "latency_events": 2,
            "depth_multiplier": 0.55,
            "fill_intensity_multiplier": 0.65,
            "adverse_selection_multiplier": 0.95,
            "midpoint": False,
            "min_fill_size": 1.0,
            "eligible_market": True,
            "eligible_limit": True,
        },
        {
            "venue": "DARK_MID",
            "venue_type": "dark_midpoint",
            "taker_fee_bps": 0.10,
            "maker_rebate_bps": 0.00,
            "latency_events": 5,
            "depth_multiplier": 0.35,
            "fill_intensity_multiplier": 0.42,
            "adverse_selection_multiplier": 0.75,
            "midpoint": True,
            "min_fill_size": 300.0,
            "eligible_market": False,
            "eligible_limit": True,
        },
        {
            "venue": "FAST",
            "venue_type": "lit_fast",
            "taker_fee_bps": 0.50,
            "maker_rebate_bps": -0.02,
            "latency_events": 0,
            "depth_multiplier": 0.65,
            "fill_intensity_multiplier": 0.90,
            "adverse_selection_multiplier": 1.15,
            "midpoint": False,
            "min_fill_size": 1.0,
            "eligible_market": True,
            "eligible_limit": True,
        },
    ])
    return venues

venues = make_venue_metadata()
venue_names = venues["venue"].tolist()

venues

## 6. Simulate consolidated market state

We simulate a consolidated mid-price, spread, volatility regime, and marketable flow.

Venue-specific quotes are then generated by adding depth and latency differences.

In [ ]:
def round_to_tick(x, tick):
    return np.round(x / tick) * tick

def simulate_consolidated_market(config):
    rng = np.random.default_rng(config.seed)
    n = config.n_events

    event_id = np.arange(n)

    stress = np.zeros(n, dtype=int)
    start = int(config.n_events * config.stress_start_frac)
    end = int(config.n_events * config.stress_end_frac)
    stress[start:end] = 1

    vol = np.where(stress == 1, 0.00035, 0.00013)
    returns = rng.standard_t(df=6, size=n) * vol
    mid = config.initial_mid * np.exp(np.cumsum(returns))
    mid = round_to_tick(mid, config.tick_size)

    spread_ticks = rng.choice(np.arange(1, 7), size=n, p=[0.34, 0.28, 0.16, 0.10, 0.07, 0.05])
    spread_ticks = spread_ticks + stress * rng.integers(1, 5, size=n)
    spread = spread_ticks * config.tick_size

    bid = round_to_tick(mid - spread / 2, config.tick_size)
    ask = round_to_tick(mid + spread / 2, config.tick_size)
    ask = np.maximum(ask, bid + config.tick_size)

    # Flow imbalance AR process.
    latent_imbalance = np.zeros(n)
    rng2 = np.random.default_rng(config.seed + 99)
    for t in range(1, n):
        latent_imbalance[t] = 0.97 * latent_imbalance[t - 1] + rng2.normal(0, 0.10)
        latent_imbalance[t] += 0.15 * stress[t]

    buy_pressure = 1.0 / (1.0 + np.exp(-latent_imbalance))
    buy_mkt_volume = rng.lognormal(np.log(700), 0.70, n) * buy_pressure * np.where(stress == 1, 1.5, 1.0)
    sell_mkt_volume = rng.lognormal(np.log(700), 0.70, n) * (1.0 - buy_pressure) * np.where(stress == 1, 1.5, 1.0)

    market = pd.DataFrame({
        "event_id": event_id,
        "stress": stress,
        "mid": mid,
        "bid": bid,
        "ask": ask,
        "spread": ask - bid,
        "spread_ticks": (ask - bid) / config.tick_size,
        "return": np.r_[0.0, np.diff(np.log(mid + 1e-12))],
        "buy_mkt_volume": buy_mkt_volume,
        "sell_mkt_volume": sell_mkt_volume,
        "latent_imbalance": latent_imbalance,
    })

    market["realised_vol_200"] = market["return"].rolling(200).std().bfill().fillna(0.0)
    market["flow_imbalance_100"] = (
        market["buy_mkt_volume"].rolling(100).sum()
        - market["sell_mkt_volume"].rolling(100).sum()
    ) / (
        market["buy_mkt_volume"].rolling(100).sum()
        + market["sell_mkt_volume"].rolling(100).sum()
    )
    market["flow_imbalance_100"] = market["flow_imbalance_100"].fillna(0.0)

    return market

market = simulate_consolidated_market(config)

market.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(market["event_id"], market["mid"])
plt.title("Consolidated Mid Price")
plt.xlabel("Event")
plt.ylabel("Mid")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(market["event_id"], market["spread_ticks"], label="spread ticks")
plt.plot(market["event_id"], market["stress"] * market["spread_ticks"].max(), label="stress marker", alpha=0.4)
plt.title("Spread and Stress Regime")
plt.xlabel("Event")
plt.legend()
plt.show()

## 7. Venue-specific quote state

Each venue has:

- bid/ask price;
- visible bid/ask depth;
- venue-specific latency;
- adverse selection proxy;
- midpoint fill probability for dark venue.

Lit venue quotes are generated around consolidated best bid/ask with small idiosyncratic variation.

In [ ]:
def simulate_venue_books(market, venues, config):
    rng = np.random.default_rng(config.seed + 222)
    rows = []

    for _, venue in venues.iterrows():
        v = venue["venue"]
        depth_mult = venue["depth_multiplier"]

        # Venue-specific micro-spread adjustment.
        quote_noise_ticks = rng.choice([-1, 0, 1], size=len(market), p=[0.12, 0.76, 0.12])
        bid = market["bid"].to_numpy() - np.maximum(quote_noise_ticks, 0) * config.tick_size
        ask = market["ask"].to_numpy() + np.maximum(-quote_noise_ticks, 0) * config.tick_size

        if venue["midpoint"]:
            bid = market["mid"].to_numpy()
            ask = market["mid"].to_numpy()

        stress_depth_mult = np.where(market["stress"].to_numpy() == 1, 0.45, 1.0)
        bid_depth = rng.lognormal(np.log(2_800 * depth_mult), 0.60, len(market)) * stress_depth_mult
        ask_depth = rng.lognormal(np.log(2_800 * depth_mult), 0.60, len(market)) * stress_depth_mult

        for i in range(len(market)):
            rows.append({
                "event_id": int(market.iloc[i]["event_id"]),
                "venue": v,
                "venue_type": venue["venue_type"],
                "bid": bid[i],
                "ask": ask[i],
                "mid": market.iloc[i]["mid"],
                "bid_depth": bid_depth[i],
                "ask_depth": ask_depth[i],
                "spread": ask[i] - bid[i],
                "spread_ticks": (ask[i] - bid[i]) / config.tick_size,
                "stress": int(market.iloc[i]["stress"]),
                "realised_vol_200": market.iloc[i]["realised_vol_200"],
                "flow_imbalance_100": market.iloc[i]["flow_imbalance_100"],
                "buy_mkt_volume": market.iloc[i]["buy_mkt_volume"] * venue["fill_intensity_multiplier"],
                "sell_mkt_volume": market.iloc[i]["sell_mkt_volume"] * venue["fill_intensity_multiplier"],
            })

    venue_books = pd.DataFrame(rows)
    return venue_books

venue_books = simulate_venue_books(market, venues, config)

venue_books.head()

## 8. Generate parent orders

Each parent order is:

- buy or sell;
- event time;
- target quantity;
- urgency;
- max child orders;
- benchmark arrival mid.

The router breaks each parent into child orders.

In [ ]:
def generate_parent_orders(market, config):
    rng = np.random.default_rng(config.seed + 333)

    submit_events = np.sort(
        rng.choice(
            np.arange(100, config.n_events - 300),
            size=config.n_parent_orders,
            replace=False,
        )
    )

    side = rng.choice(np.array([1, -1]), size=config.n_parent_orders)
    quantity = rng.lognormal(np.log(config.parent_order_median_size), 0.75, config.n_parent_orders)
    quantity = np.maximum(config.child_order_size, np.round(quantity / 100) * 100)

    urgency = rng.choice(["low", "medium", "high"], size=config.n_parent_orders, p=[0.35, 0.45, 0.20])

    parents = pd.DataFrame({
        "parent_id": np.arange(config.n_parent_orders, dtype=int),
        "submit_event": submit_events.astype(int),
        "side": side.astype(int),
        "side_name": np.where(side > 0, "BUY", "SELL"),
        "quantity": quantity.astype(float),
        "urgency": urgency,
    })

    parents["arrival_mid"] = market.set_index("event_id").loc[parents["submit_event"], "mid"].to_numpy()
    parents["max_child_orders"] = np.minimum(
        config.max_child_orders_per_parent,
        np.ceil(parents["quantity"] / config.child_order_size).astype(int),
    )

    return parents

parents = generate_parent_orders(market, config)

parents.head()

## 9. Venue feature snapshot

At a parent order event, the router sees a delayed venue state due to latency:

$$
observedEvent_v = submitEvent - latency_v
$$

The router uses that observed state to score venues.

Actual execution occurs after latency.

In [ ]:
def get_venue_snapshot(parent, market, venue_books, venues, config):
    rows = []
    submit = int(parent["submit_event"])
    side = int(parent["side"])

    vb_indexed = venue_books.set_index(["event_id", "venue"])

    for _, venue in venues.iterrows():
        v = venue["venue"]
        latency = int(venue["latency_events"])
        observed_event = max(0, submit - latency)
        execution_event = min(config.n_events - 1, submit + latency)

        obs = vb_indexed.loc[(observed_event, v)]
        exe = vb_indexed.loc[(execution_event, v)]

        if side > 0:
            touch_price = obs["ask"]
            depth = obs["ask_depth"]
            contra_flow = obs["sell_mkt_volume"]
            same_flow = obs["buy_mkt_volume"]
        else:
            touch_price = obs["bid"]
            depth = obs["bid_depth"]
            contra_flow = obs["buy_mkt_volume"]
            same_flow = obs["sell_mkt_volume"]

        rows.append({
            "parent_id": int(parent["parent_id"]),
            "venue": v,
            "venue_type": venue["venue_type"],
            "side": side,
            "observed_event": observed_event,
            "execution_event": execution_event,
            "latency_events": latency,
            "observed_mid": obs["mid"],
            "observed_bid": obs["bid"],
            "observed_ask": obs["ask"],
            "execution_bid": exe["bid"],
            "execution_ask": exe["ask"],
            "execution_mid": exe["mid"],
            "touch_price_observed": touch_price,
            "available_depth_observed": depth,
            "spread_ticks_observed": obs["spread_ticks"],
            "flow_imbalance_100": obs["flow_imbalance_100"],
            "realised_vol_200": obs["realised_vol_200"],
            "stress": int(obs["stress"]),
            "contra_flow_proxy": contra_flow,
            "same_flow_proxy": same_flow,
            "taker_fee_bps": venue["taker_fee_bps"],
            "maker_rebate_bps": venue["maker_rebate_bps"],
            "fill_intensity_multiplier": venue["fill_intensity_multiplier"],
            "adverse_selection_multiplier": venue["adverse_selection_multiplier"],
            "midpoint": bool(venue["midpoint"]),
            "min_fill_size": venue["min_fill_size"],
            "eligible_market": bool(venue["eligible_market"]),
            "eligible_limit": bool(venue["eligible_limit"]),
        })

    return pd.DataFrame(rows)

snapshot_example = get_venue_snapshot(parents.iloc[0], market, venue_books, venues, config)

snapshot_example

## 10. Venue scoring functions

We compute expected costs for marketable and passive routing.

### Marketable expected cost

For buys:

$$
cost = \frac{executionPrice-arrivalMid}{arrivalMid} + fee + impact + adverse
$$

For sells:

$$
cost = \frac{arrivalMid-executionPrice}{arrivalMid} + fee + impact + adverse
$$

### Passive expected value

Passive routing accounts for:

- fill probability;
- spread capture;
- maker rebate;
- adverse selection;
- opportunity cost if not filled.

In [ ]:
def estimate_market_impact_bps(child_qty, depth, stress):
    depth = max(depth, 1.0)
    participation = child_qty / depth
    base = 0.75 * np.sqrt(max(participation, 0.0))
    stress_mult = 1.75 if stress else 1.0
    return base * stress_mult

def estimate_adverse_selection_bps(row, side, passive=True):
    imbalance = row["flow_imbalance_100"]
    vol_bps = row["realised_vol_200"] * 10000
    side_pressure = imbalance if side > 0 else -imbalance
    base = max(0.0, side_pressure) * 0.8 + vol_bps * 0.15
    if passive:
        base *= 1.10
    return base * row["adverse_selection_multiplier"]

def estimate_passive_fill_probability(row, side, child_qty, time_in_force_events, distance_ticks=0):
    depth = max(row["available_depth_observed"], 1.0)
    queue_needed = depth * (1.0 + 0.35 * distance_ticks) + child_qty

    if side > 0:
        favourable_flow = max(row["contra_flow_proxy"], 0.0)
        side_adj_imbalance = -row["flow_imbalance_100"]
    else:
        favourable_flow = max(row["contra_flow_proxy"], 0.0)
        side_adj_imbalance = row["flow_imbalance_100"]

    flow_rate = favourable_flow / max(depth, 1.0)
    latency_penalty = np.exp(-0.05 * row["latency_events"])
    spread_bonus = 1.0 + 0.04 * row["spread_ticks_observed"]
    imbalance_bonus = np.exp(0.75 * side_adj_imbalance)
    stress_bonus = 1.20 if row["stress"] else 1.0

    hazard = (
        0.012
        * flow_rate
        * time_in_force_events
        * latency_penalty
        * spread_bonus
        * imbalance_bonus
        * stress_bonus
        * row["fill_intensity_multiplier"]
    )

    distance_penalty = np.exp(-0.55 * distance_ticks)
    size_penalty = np.exp(-0.15 * child_qty / max(depth, 1.0))

    prob = 1.0 - np.exp(-hazard * distance_penalty * size_penalty)
    return float(np.clip(prob, 0.0, 1.0))

def score_venue_for_child(parent, venue_row, child_qty, config, mode="market"):
    side = int(parent["side"])
    arrival_mid = float(parent["arrival_mid"])

    if mode == "market":
        if not venue_row["eligible_market"]:
            return np.inf, 0.0, {"reason": "not_market_eligible"}

        if side > 0:
            exec_price = venue_row["execution_ask"]
            price_cost_bps = (exec_price - arrival_mid) / arrival_mid * 10000.0
            depth = venue_row["available_depth_observed"]
        else:
            exec_price = venue_row["execution_bid"]
            price_cost_bps = (arrival_mid - exec_price) / arrival_mid * 10000.0
            depth = venue_row["available_depth_observed"]

        impact_bps = estimate_market_impact_bps(child_qty, depth, venue_row["stress"])
        adverse_bps = estimate_adverse_selection_bps(venue_row, side, passive=False)
        fee_bps = venue_row["taker_fee_bps"]

        total_cost_bps = price_cost_bps + fee_bps + impact_bps + adverse_bps

        details = {
            "price_cost_bps": price_cost_bps,
            "fee_bps": fee_bps,
            "impact_bps": impact_bps,
            "adverse_bps": adverse_bps,
            "fill_prob": 1.0,
            "expected_cost_bps": total_cost_bps,
            "reason": "market_score",
        }

        return total_cost_bps, 1.0, details

    if mode == "passive":
        if not venue_row["eligible_limit"]:
            return np.inf, 0.0, {"reason": "not_limit_eligible"}

        if venue_row["midpoint"]:
            distance_ticks = 0
            if child_qty < venue_row["min_fill_size"]:
                return np.inf, 0.0, {"reason": "below_min_fill"}
            spread_capture_bps = venue_row["spread_ticks_observed"] * config.tick_size / max(arrival_mid, 1e-12) * 10000.0 / 2.0
        else:
            distance_ticks = 0
            spread_capture_bps = venue_row["spread_ticks_observed"] * config.tick_size / max(arrival_mid, 1e-12) * 10000.0 / 2.0

        fill_prob = estimate_passive_fill_probability(
            venue_row,
            side=side,
            child_qty=child_qty,
            time_in_force_events=config.time_in_force_events,
            distance_ticks=distance_ticks,
        )

        adverse_bps = estimate_adverse_selection_bps(venue_row, side, passive=True)
        rebate_bps = venue_row["maker_rebate_bps"]
        opportunity_bps = config.opportunity_cost_bps * (1.0 - fill_prob)

        expected_value_bps = fill_prob * (spread_capture_bps - adverse_bps - rebate_bps) - opportunity_bps
        expected_cost_bps = -expected_value_bps

        details = {
            "spread_capture_bps": spread_capture_bps,
            "rebate_bps": rebate_bps,
            "adverse_bps": adverse_bps,
            "opportunity_bps": opportunity_bps,
            "fill_prob": fill_prob,
            "expected_cost_bps": expected_cost_bps,
            "reason": "passive_score",
        }

        return expected_cost_bps, fill_prob, details

    raise ValueError("mode must be market or passive")

## 11. Routing policies

The router returns child-order instructions:

- venue;
- mode: market or passive;
- quantity;
- expected cost;
- fill probability.

Different policies choose venues differently.

In [ ]:
def split_quantity(quantity, child_size, max_children):
    n_child = int(min(max_children, np.ceil(quantity / child_size)))
    if n_child <= 0:
        return []
    base = quantity / n_child
    return [base for _ in range(n_child)]

def route_child(parent, snapshot, child_qty, policy, config):
    rows = []

    if policy == "primary_only":
        row = snapshot[snapshot["venue"] == "PRIMARY"].iloc[0]
        cost, prob, details = score_venue_for_child(parent, row, child_qty, config, mode="market")
        return [{
            "venue": row["venue"],
            "mode": "market",
            "quantity": child_qty,
            "expected_cost_bps": cost,
            "fill_prob": prob,
            **details,
        }]

    if policy == "cheapest_taker":
        scored = []
        for _, row in snapshot.iterrows():
            cost, prob, details = score_venue_for_child(parent, row, child_qty, config, mode="market")
            scored.append((cost, row, prob, details))
        cost, row, prob, details = min(scored, key=lambda x: x[0])
        return [{
            "venue": row["venue"],
            "mode": "market",
            "quantity": child_qty,
            "expected_cost_bps": cost,
            "fill_prob": prob,
            **details,
        }]

    if policy == "fastest":
        eligible = snapshot[snapshot["eligible_market"]].sort_values("latency_events")
        row = eligible.iloc[0]
        cost, prob, details = score_venue_for_child(parent, row, child_qty, config, mode="market")
        return [{
            "venue": row["venue"],
            "mode": "market",
            "quantity": child_qty,
            "expected_cost_bps": cost,
            "fill_prob": prob,
            **details,
        }]

    if policy == "liquidity_weighted":
        eligible = snapshot[snapshot["eligible_market"]].copy()
        weights = eligible["available_depth_observed"].clip(lower=1.0)
        weights = weights / weights.sum()
        instructions = []
        for (_, row), w in zip(eligible.iterrows(), weights):
            q = child_qty * w
            cost, prob, details = score_venue_for_child(parent, row, q, config, mode="market")
            instructions.append({
                "venue": row["venue"],
                "mode": "market",
                "quantity": q,
                "expected_cost_bps": cost,
                "fill_prob": prob,
                **details,
            })
        return instructions

    if policy == "expected_cost":
        scored = []
        for _, row in snapshot.iterrows():
            for mode in ["market", "passive"]:
                cost, prob, details = score_venue_for_child(parent, row, child_qty, config, mode=mode)
                scored.append((cost, row, mode, prob, details))
        cost, row, mode, prob, details = min(scored, key=lambda x: x[0])
        return [{
            "venue": row["venue"],
            "mode": mode,
            "quantity": child_qty,
            "expected_cost_bps": cost,
            "fill_prob": prob,
            **details,
        }]

    if policy == "fill_probability_aware":
        # Choose passive only if expected cost is good and fill probability is not terrible; otherwise market.
        candidates = []
        for _, row in snapshot.iterrows():
            pcost, pprob, pdetails = score_venue_for_child(parent, row, child_qty, config, mode="passive")
            mcost, mprob, mdetails = score_venue_for_child(parent, row, child_qty, config, mode="market")
            if np.isfinite(pcost) and pprob > 0.35 and pcost < mcost:
                candidates.append((pcost, row, "passive", pprob, pdetails))
            elif np.isfinite(mcost):
                candidates.append((mcost, row, "market", mprob, mdetails))
        cost, row, mode, prob, details = min(candidates, key=lambda x: x[0])
        return [{
            "venue": row["venue"],
            "mode": mode,
            "quantity": child_qty,
            "expected_cost_bps": cost,
            "fill_prob": prob,
            **details,
        }]

    if policy == "dark_first":
        dark = snapshot[snapshot["midpoint"]]
        instructions = []
        if len(dark):
            row = dark.iloc[0]
            dark_qty = min(child_qty * 0.50, child_qty)
            cost, prob, details = score_venue_for_child(parent, row, dark_qty, config, mode="passive")
            if np.isfinite(cost):
                instructions.append({
                    "venue": row["venue"],
                    "mode": "passive",
                    "quantity": dark_qty,
                    "expected_cost_bps": cost,
                    "fill_prob": prob,
                    **details,
                })
                residual_qty = child_qty - dark_qty
            else:
                residual_qty = child_qty
        else:
            residual_qty = child_qty

        if residual_qty > 1e-9 and config.fallback_cross_after_dark:
            lit = snapshot[snapshot["eligible_market"]].copy()
            scored = []
            for _, row in lit.iterrows():
                cost, prob, details = score_venue_for_child(parent, row, residual_qty, config, mode="market")
                scored.append((cost, row, prob, details))
            cost, row, prob, details = min(scored, key=lambda x: x[0])
            instructions.append({
                "venue": row["venue"],
                "mode": "market",
                "quantity": residual_qty,
                "expected_cost_bps": cost,
                "fill_prob": prob,
                **details,
            })
        return instructions

    raise ValueError(f"Unknown policy {policy}")

policies = [
    "primary_only",
    "cheapest_taker",
    "fastest",
    "liquidity_weighted",
    "expected_cost",
    "fill_probability_aware",
    "dark_first",
]

example_snapshot = get_venue_snapshot(parents.iloc[10], market, venue_books, venues, config)
route_child(parents.iloc[10], example_snapshot, 1_000, "expected_cost", config)

## 12. Simulate child order fills

Rules:

### Market child

- fills immediately at execution quote;
- capped by venue depth;
- incurs taker fee;
- pays impact and adverse-selection proxy.

### Passive child

- fills probabilistically;
- if filled, earns spread/midpoint improvement;
- if not filled, may incur opportunity cost;
- can fall back depending on policy.

In [ ]:
def simulate_child_fill(parent, instruction, snapshot_row, config, rng):
    side = int(parent["side"])
    q = float(instruction["quantity"])
    arrival_mid = float(parent["arrival_mid"])
    venue = instruction["venue"]
    mode = instruction["mode"]

    if mode == "market":
        if side > 0:
            exec_price = snapshot_row["execution_ask"]
            price_cost_bps = (exec_price - arrival_mid) / arrival_mid * 10000.0
            depth = snapshot_row["available_depth_observed"]
        else:
            exec_price = snapshot_row["execution_bid"]
            price_cost_bps = (arrival_mid - exec_price) / arrival_mid * 10000.0
            depth = snapshot_row["available_depth_observed"]

        filled_qty = min(q, max(depth, 0.0))
        fill_prob = 1.0 if filled_qty > 0 else 0.0
        unfilled_qty = q - filled_qty

        fee_bps = snapshot_row["taker_fee_bps"]
        impact_bps = estimate_market_impact_bps(filled_qty, depth, snapshot_row["stress"])
        adverse_bps = estimate_adverse_selection_bps(snapshot_row, side, passive=False)
        realised_cost_bps = price_cost_bps + fee_bps + impact_bps + adverse_bps

        return {
            "venue": venue,
            "mode": mode,
            "requested_qty": q,
            "filled_qty": filled_qty,
            "unfilled_qty": unfilled_qty,
            "fill_prob_model": instruction["fill_prob"],
            "filled": int(filled_qty > 0),
            "realised_cost_bps": realised_cost_bps,
            "fee_bps": fee_bps,
            "impact_bps": impact_bps,
            "adverse_bps": adverse_bps,
            "price_cost_bps": price_cost_bps,
            "opportunity_cost_bps": 0.0 if filled_qty > 0 else config.opportunity_cost_bps,
        }

    if mode == "passive":
        p = float(np.clip(instruction["fill_prob"], 0.0, 1.0))
        filled = rng.uniform() < p

        if not filled:
            return {
                "venue": venue,
                "mode": mode,
                "requested_qty": q,
                "filled_qty": 0.0,
                "unfilled_qty": q,
                "fill_prob_model": p,
                "filled": 0,
                "realised_cost_bps": config.opportunity_cost_bps,
                "fee_bps": 0.0,
                "impact_bps": 0.0,
                "adverse_bps": 0.0,
                "price_cost_bps": 0.0,
                "opportunity_cost_bps": config.opportunity_cost_bps,
            }

        if snapshot_row["midpoint"]:
            spread_capture_bps = snapshot_row["spread_ticks_observed"] * config.tick_size / arrival_mid * 10000.0 / 2.0
        else:
            spread_capture_bps = snapshot_row["spread_ticks_observed"] * config.tick_size / arrival_mid * 10000.0 / 2.0

        rebate_bps = snapshot_row["maker_rebate_bps"]
        adverse_bps = estimate_adverse_selection_bps(snapshot_row, side, passive=True)
        realised_cost_bps = -spread_capture_bps + rebate_bps + adverse_bps

        return {
            "venue": venue,
            "mode": mode,
            "requested_qty": q,
            "filled_qty": q,
            "unfilled_qty": 0.0,
            "fill_prob_model": p,
            "filled": 1,
            "realised_cost_bps": realised_cost_bps,
            "fee_bps": rebate_bps,
            "impact_bps": 0.0,
            "adverse_bps": adverse_bps,
            "price_cost_bps": -spread_capture_bps,
            "opportunity_cost_bps": 0.0,
        }

    raise ValueError("Unknown mode")

## 13. Parent-order routing simulator

For each parent order:

1. split into child quantities;
2. route each child according to policy;
3. simulate fills;
4. aggregate parent-level cost and fill quality.

In [ ]:
def simulate_routing_policy(parents, market, venue_books, venues, policy, config):
    rng = np.random.default_rng(config.seed + abs(hash(policy)) % 100_000)

    child_rows = []
    parent_rows = []

    vb_index = venue_books.set_index(["event_id", "venue"])

    for _, parent in parents.iterrows():
        child_qtys = split_quantity(parent["quantity"], config.child_order_size, int(parent["max_child_orders"]))
        snapshot = get_venue_snapshot(parent, market, venue_books, venues, config)
        snapshot_by_venue = snapshot.set_index("venue")

        parent_requested = 0.0
        parent_filled = 0.0
        weighted_cost_sum = 0.0
        expected_cost_sum = 0.0
        child_count = 0
        passive_count = 0
        market_count = 0

        for child_idx, child_qty in enumerate(child_qtys):
            instructions = route_child(parent, snapshot, child_qty, policy, config)

            for instruction in instructions:
                venue = instruction["venue"]
                snap_row = snapshot_by_venue.loc[venue]
                fill = simulate_child_fill(parent, instruction, snap_row, config, rng)

                requested = fill["requested_qty"]
                filled = fill["filled_qty"]
                parent_requested += requested
                parent_filled += filled
                weighted_cost_sum += fill["realised_cost_bps"] * requested
                expected_cost_sum += instruction["expected_cost_bps"] * requested
                child_count += 1

                if instruction["mode"] == "passive":
                    passive_count += 1
                else:
                    market_count += 1

                child_rows.append({
                    "parent_id": int(parent["parent_id"]),
                    "child_idx": child_idx,
                    "policy": policy,
                    "side": int(parent["side"]),
                    "side_name": parent["side_name"],
                    "urgency": parent["urgency"],
                    "submit_event": int(parent["submit_event"]),
                    "venue": venue,
                    "mode": instruction["mode"],
                    **fill,
                    "expected_cost_bps": instruction["expected_cost_bps"],
                })

        avg_realised_cost = weighted_cost_sum / max(parent_requested, 1e-12)
        avg_expected_cost = expected_cost_sum / max(parent_requested, 1e-12)
        fill_rate = parent_filled / max(parent_requested, 1e-12)

        parent_rows.append({
            "parent_id": int(parent["parent_id"]),
            "policy": policy,
            "side": int(parent["side"]),
            "side_name": parent["side_name"],
            "urgency": parent["urgency"],
            "submit_event": int(parent["submit_event"]),
            "arrival_mid": parent["arrival_mid"],
            "requested_qty": parent_requested,
            "filled_qty": parent_filled,
            "fill_rate": fill_rate,
            "avg_realised_cost_bps": avg_realised_cost,
            "avg_expected_cost_bps": avg_expected_cost,
            "expected_minus_realised_bps": avg_expected_cost - avg_realised_cost,
            "child_count": child_count,
            "passive_child_count": passive_count,
            "market_child_count": market_count,
            "passive_child_rate": passive_count / max(child_count, 1),
        })

    return pd.DataFrame(parent_rows), pd.DataFrame(child_rows)

routing_outputs = {}
for policy in policies:
    routing_outputs[policy] = simulate_routing_policy(parents, market, venue_books, venues, policy, config)

parent_primary, child_primary = routing_outputs["primary_only"]
parent_primary.head()

## 14. Routing policy comparison

In [ ]:
def summarise_routing(parent_df, child_df):
    summary = (
        parent_df
        .groupby("policy")
        .agg(
            n_parents=("parent_id", "count"),
            mean_fill_rate=("fill_rate", "mean"),
            p05_fill_rate=("fill_rate", lambda x: x.quantile(0.05)),
            mean_realised_cost_bps=("avg_realised_cost_bps", "mean"),
            median_realised_cost_bps=("avg_realised_cost_bps", "median"),
            p95_realised_cost_bps=("avg_realised_cost_bps", lambda x: x.quantile(0.95)),
            mean_expected_cost_bps=("avg_expected_cost_bps", "mean"),
            mean_expected_minus_realised_bps=("expected_minus_realised_bps", "mean"),
            mean_child_count=("child_count", "mean"),
            mean_passive_child_rate=("passive_child_rate", "mean"),
        )
        .reset_index()
    )

    venue_summary = (
        child_df
        .groupby(["policy", "venue", "mode"])
        .agg(
            n_children=("parent_id", "count"),
            requested_qty=("requested_qty", "sum"),
            filled_qty=("filled_qty", "sum"),
            mean_fill_rate=("filled", "mean"),
            mean_realised_cost_bps=("realised_cost_bps", "mean"),
            mean_expected_cost_bps=("expected_cost_bps", "mean"),
        )
        .reset_index()
    )

    return summary, venue_summary

all_parent = pd.concat([routing_outputs[p][0] for p in policies], ignore_index=True)
all_child = pd.concat([routing_outputs[p][1] for p in policies], ignore_index=True)

policy_summary, venue_tca = summarise_routing(all_parent, all_child)

policy_summary.sort_values("mean_realised_cost_bps")

In [ ]:
plt.figure(figsize=(11, 5))
plot_df = policy_summary.sort_values("mean_realised_cost_bps")
plt.barh(plot_df["policy"], plot_df["mean_realised_cost_bps"])
plt.title("Mean Realised Cost by Routing Policy")
plt.xlabel("Realised cost, bps")
plt.ylabel("Policy")
plt.show()

plt.figure(figsize=(11, 5))
plot_df = policy_summary.sort_values("mean_fill_rate")
plt.barh(plot_df["policy"], plot_df["mean_fill_rate"])
plt.title("Mean Fill Rate by Routing Policy")
plt.xlabel("Fill rate")
plt.ylabel("Policy")
plt.show()

## 15. Venue-level transaction cost analysis

Venue TCA answers:

- where did orders go?
- which venues filled?
- where were costs high?
- where did expected cost differ from realised cost?

In [ ]:
venue_tca.sort_values(["policy", "venue", "mode"]).head(30)

In [ ]:
selected_policy = "expected_cost"
selected_venue = venue_tca[venue_tca["policy"] == selected_policy].copy()

plt.figure(figsize=(10, 5))
for mode, group in selected_venue.groupby("mode"):
    plt.bar(group["venue"] + "_" + mode, group["requested_qty"], label=mode)
plt.title(f"Requested Quantity by Venue and Mode — {selected_policy}")
plt.xlabel("Venue / mode")
plt.ylabel("Requested quantity")
plt.xticks(rotation=45)
plt.show()

## 16. Stress-regime performance

Routers often behave differently in stress:

- spreads widen;
- depth falls;
- passive fills may rise or become toxic;
- market orders pay more.

We compare normal and stress periods.

In [ ]:
parent_with_regime = all_parent.merge(
    market[["event_id", "stress", "spread_ticks", "realised_vol_200"]],
    left_on="submit_event",
    right_on="event_id",
    how="left",
)

stress_performance = (
    parent_with_regime
    .groupby(["policy", "stress"])
    .agg(
        n_parents=("parent_id", "count"),
        mean_fill_rate=("fill_rate", "mean"),
        mean_realised_cost_bps=("avg_realised_cost_bps", "mean"),
        p95_realised_cost_bps=("avg_realised_cost_bps", lambda x: x.quantile(0.95)),
        mean_passive_child_rate=("passive_child_rate", "mean"),
        mean_spread_ticks=("spread_ticks", "mean"),
    )
    .reset_index()
)

stress_performance.head(20)

## 17. Expected versus realised cost diagnostics

The router should not systematically underestimate cost.

We inspect prediction error:

$$
error = expectedCost - realisedCost
$$

Large positive error means expected cost is too pessimistic.

Large negative error means expected cost is too optimistic.

In [ ]:
cost_error_report = (
    all_parent
    .groupby("policy")
    .agg(
        mean_error_bps=("expected_minus_realised_bps", "mean"),
        median_error_bps=("expected_minus_realised_bps", "median"),
        p05_error_bps=("expected_minus_realised_bps", lambda x: x.quantile(0.05)),
        p95_error_bps=("expected_minus_realised_bps", lambda x: x.quantile(0.95)),
        mae_error_bps=("expected_minus_realised_bps", lambda x: np.mean(np.abs(x))),
    )
    .reset_index()
)

cost_error_report

In [ ]:
plt.figure(figsize=(10, 5))
for policy in ["expected_cost", "fill_probability_aware", "dark_first"]:
    sub = all_parent[all_parent["policy"] == policy]
    plt.hist(sub["expected_minus_realised_bps"], bins=60, alpha=0.45, label=policy)
plt.axvline(0, linestyle="--")
plt.title("Expected Minus Realised Cost")
plt.xlabel("Error, bps")
plt.ylabel("Count")
plt.legend()
plt.show()

## 18. Venue allocation optimisation

A simple brute-force split optimiser searches allocations across eligible market venues.

For each split vector $w_v$:

$$
\sum_v w_v = 1
$$

Expected cost is:

$$
\sum_v w_v cost_v(w_v Q)
$$

This captures that impact depends on how much size is sent to each venue.

In [ ]:
def brute_force_market_split(parent, snapshot, child_qty, config, grid_step=0.25):
    eligible = snapshot[snapshot["eligible_market"]].copy().reset_index(drop=True)
    venue_list = eligible["venue"].tolist()
    n = len(eligible)

    if n == 0:
        return []

    # Recursive grid simplex for small n.
    allocations = []

    def build(prefix, remaining, k):
        if k == n - 1:
            allocations.append(prefix + [remaining])
            return
        steps = int(round(remaining / grid_step))
        for s in range(steps + 1):
            w = s * grid_step
            build(prefix + [w], remaining - w, k + 1)

    build([], 1.0, 0)

    best = None
    best_details = None

    for alloc in allocations:
        total_cost = 0.0
        valid = True
        details = []
        for w, (_, row) in zip(alloc, eligible.iterrows()):
            q = child_qty * w
            if q <= 1e-12:
                continue
            cost, prob, d = score_venue_for_child(parent, row, q, config, mode="market")
            if not np.isfinite(cost):
                valid = False
                break
            total_cost += w * cost
            details.append((row["venue"], q, cost, prob, d))

        if not valid:
            continue

        if best is None or total_cost < best:
            best = total_cost
            best_details = details

    if best_details is None:
        return route_child(parent, snapshot, child_qty, "cheapest_taker", config)

    instructions = []
    for venue, q, cost, prob, details in best_details:
        instructions.append({
            "venue": venue,
            "mode": "market",
            "quantity": q,
            "expected_cost_bps": cost,
            "fill_prob": prob,
            **details,
        })
    return instructions

def route_child_with_optimised_split(parent, snapshot, child_qty, config):
    return brute_force_market_split(parent, snapshot, child_qty, config, grid_step=0.25)

# Demonstration on one parent.
demo_parent = parents.iloc[20]
demo_snapshot = get_venue_snapshot(demo_parent, market, venue_books, venues, config)
route_child_with_optimised_split(demo_parent, demo_snapshot, 2_500, config)

## 19. Optimised-split policy simulation

Add an optimiser-based market split policy and compare.

In [ ]:
def simulate_optimised_split_policy(parents, market, venue_books, venues, config):
    rng = np.random.default_rng(config.seed + 9090)
    child_rows = []
    parent_rows = []

    for _, parent in parents.iterrows():
        child_qtys = split_quantity(parent["quantity"], config.child_order_size, int(parent["max_child_orders"]))
        snapshot = get_venue_snapshot(parent, market, venue_books, venues, config)
        snapshot_by_venue = snapshot.set_index("venue")

        parent_requested = 0.0
        parent_filled = 0.0
        weighted_cost_sum = 0.0
        expected_cost_sum = 0.0
        child_count = 0

        for child_idx, child_qty in enumerate(child_qtys):
            instructions = route_child_with_optimised_split(parent, snapshot, child_qty, config)

            for instruction in instructions:
                venue = instruction["venue"]
                snap_row = snapshot_by_venue.loc[venue]
                fill = simulate_child_fill(parent, instruction, snap_row, config, rng)

                parent_requested += fill["requested_qty"]
                parent_filled += fill["filled_qty"]
                weighted_cost_sum += fill["realised_cost_bps"] * fill["requested_qty"]
                expected_cost_sum += instruction["expected_cost_bps"] * fill["requested_qty"]
                child_count += 1

                child_rows.append({
                    "parent_id": int(parent["parent_id"]),
                    "child_idx": child_idx,
                    "policy": "optimised_split",
                    "side": int(parent["side"]),
                    "side_name": parent["side_name"],
                    "urgency": parent["urgency"],
                    "submit_event": int(parent["submit_event"]),
                    "venue": venue,
                    "mode": instruction["mode"],
                    **fill,
                    "expected_cost_bps": instruction["expected_cost_bps"],
                })

        parent_rows.append({
            "parent_id": int(parent["parent_id"]),
            "policy": "optimised_split",
            "side": int(parent["side"]),
            "side_name": parent["side_name"],
            "urgency": parent["urgency"],
            "submit_event": int(parent["submit_event"]),
            "arrival_mid": parent["arrival_mid"],
            "requested_qty": parent_requested,
            "filled_qty": parent_filled,
            "fill_rate": parent_filled / max(parent_requested, 1e-12),
            "avg_realised_cost_bps": weighted_cost_sum / max(parent_requested, 1e-12),
            "avg_expected_cost_bps": expected_cost_sum / max(parent_requested, 1e-12),
            "expected_minus_realised_bps": expected_cost_sum / max(parent_requested, 1e-12) - weighted_cost_sum / max(parent_requested, 1e-12),
            "child_count": child_count,
            "passive_child_count": 0,
            "market_child_count": child_count,
            "passive_child_rate": 0.0,
        })

    return pd.DataFrame(parent_rows), pd.DataFrame(child_rows)

parent_opt, child_opt = simulate_optimised_split_policy(parents, market, venue_books, venues, config)

all_parent_plus = pd.concat([all_parent, parent_opt], ignore_index=True)
all_child_plus = pd.concat([all_child, child_opt], ignore_index=True)

policy_summary_plus, venue_tca_plus = summarise_routing(all_parent_plus, all_child_plus)

policy_summary_plus.sort_values("mean_realised_cost_bps")

## 20. Parent urgency diagnostics

Urgency should affect preferred routing.

High-urgency orders should generally use more marketable routes.

This synthetic router does not fully optimise urgency yet, so we report diagnostics and leave it as a governance item.

In [ ]:
urgency_report = (
    all_parent_plus
    .groupby(["policy", "urgency"])
    .agg(
        n_parents=("parent_id", "count"),
        mean_fill_rate=("fill_rate", "mean"),
        mean_realised_cost_bps=("avg_realised_cost_bps", "mean"),
        mean_passive_child_rate=("passive_child_rate", "mean"),
        p95_realised_cost_bps=("avg_realised_cost_bps", lambda x: x.quantile(0.95)),
    )
    .reset_index()
)

urgency_report.head(30)

## 21. Routing decision audit examples

Show a few parent orders where policy choices differ strongly.

In [ ]:
pivot_cost = all_parent_plus.pivot_table(
    index="parent_id",
    columns="policy",
    values="avg_realised_cost_bps",
    aggfunc="first",
)

pivot_cost["range_cost_bps"] = pivot_cost.max(axis=1) - pivot_cost.min(axis=1)
audit_examples = pivot_cost.sort_values("range_cost_bps", ascending=False).head(10)

audit_examples

## 22. Governance flags

A smart order router should be reviewed if:

1. realised cost is worse than primary-only;
2. fill rate is too low;
3. expected cost is badly miscalibrated;
4. dark routing has poor fill quality;
5. stress performance deteriorates sharply;
6. venue concentration is excessive;
7. high-urgency orders are routed passively too often;
8. synthetic data is used instead of real fills.

In [ ]:
selected_policy = "expected_cost"
selected_summary = policy_summary_plus[policy_summary_plus["policy"] == selected_policy].iloc[0]
primary_summary = policy_summary_plus[policy_summary_plus["policy"] == "primary_only"].iloc[0]

selected_parent = all_parent_plus[all_parent_plus["policy"] == selected_policy]
selected_child = all_child_plus[all_child_plus["policy"] == selected_policy]

selected_cost_error = cost_error_report[cost_error_report["policy"] == selected_policy]
selected_error_mae = selected_cost_error["mae_error_bps"].iloc[0] if len(selected_cost_error) else np.nan

dark_child = selected_child[selected_child["venue"] == "DARK_MID"]
dark_fill_rate = dark_child["filled"].mean() if len(dark_child) else np.nan
dark_mean_cost = dark_child["realised_cost_bps"].mean() if len(dark_child) else np.nan

stress_selected = stress_performance[
    (stress_performance["policy"] == selected_policy) & (stress_performance["stress"] == 1)
]
normal_selected = stress_performance[
    (stress_performance["policy"] == selected_policy) & (stress_performance["stress"] == 0)
]
if len(stress_selected) and len(normal_selected):
    stress_cost_widening = stress_selected["mean_realised_cost_bps"].iloc[0] - normal_selected["mean_realised_cost_bps"].iloc[0]
else:
    stress_cost_widening = np.nan

venue_concentration = (
    selected_child.groupby("venue")["requested_qty"].sum()
    / selected_child["requested_qty"].sum()
).max()

high_urgency_passive = selected_parent.loc[selected_parent["urgency"] == "high", "passive_child_rate"].mean()

governance_flags = pd.DataFrame([{
    "selected_policy": selected_policy,
    "selected_mean_cost_bps": selected_summary["mean_realised_cost_bps"],
    "primary_mean_cost_bps": primary_summary["mean_realised_cost_bps"],
    "selected_mean_fill_rate": selected_summary["mean_fill_rate"],
    "selected_error_mae_bps": selected_error_mae,
    "dark_fill_rate": dark_fill_rate,
    "dark_mean_cost_bps": dark_mean_cost,
    "stress_cost_widening_bps": stress_cost_widening,
    "venue_concentration_max_share": venue_concentration,
    "high_urgency_passive_rate": high_urgency_passive,
    "flag_worse_than_primary": bool(selected_summary["mean_realised_cost_bps"] > primary_summary["mean_realised_cost_bps"]),
    "flag_fill_rate_low": bool(selected_summary["mean_fill_rate"] < 0.85),
    "flag_cost_error_high": bool(selected_error_mae > 5.0) if np.isfinite(selected_error_mae) else False,
    "flag_dark_fill_rate_low": bool(dark_fill_rate < 0.25) if np.isfinite(dark_fill_rate) else False,
    "flag_stress_cost_widening_high": bool(stress_cost_widening > 5.0) if np.isfinite(stress_cost_widening) else False,
    "flag_venue_concentration_high": bool(venue_concentration > 0.75),
    "flag_high_urgency_too_passive": bool(high_urgency_passive > 0.25) if np.isfinite(high_urgency_passive) else False,
    "flag_synthetic_data_not_real_venue_fills": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_worse_than_primary",
        "flag_fill_rate_low",
        "flag_cost_error_high",
        "flag_dark_fill_rate_low",
        "flag_stress_cost_widening_high",
        "flag_venue_concentration_high",
        "flag_high_urgency_too_passive",
        "flag_synthetic_data_not_real_venue_fills",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. venue metadata is complete;
2. bid is below ask for lit venues;
3. parent quantities are positive;
4. child filled quantity never exceeds requested quantity;
5. fill rates are in $[0,1]$;
6. expected and realised costs are finite;
7. every policy generates parent outputs;
8. governance flags exist.

In [ ]:
audit_rows = []

required_venue_cols = [
    "venue", "taker_fee_bps", "maker_rebate_bps", "latency_events",
    "depth_multiplier", "eligible_market", "eligible_limit"
]
venue_complete = bool(set(required_venue_cols).issubset(venues.columns) and venues["venue"].is_unique)
audit_rows.append({
    "check": "venue_metadata_complete",
    "value": venue_complete,
    "passed": venue_complete,
})

lit_books = venue_books[venue_books["venue_type"] != "dark_midpoint"]
bid_ask_ok = bool((lit_books["bid"] < lit_books["ask"]).all())
audit_rows.append({
    "check": "lit_bid_less_than_ask",
    "value": bid_ask_ok,
    "passed": bid_ask_ok,
})

parent_qty_positive = bool((parents["quantity"] > 0).all())
audit_rows.append({
    "check": "parent_quantities_positive",
    "value": parent_qty_positive,
    "passed": parent_qty_positive,
})

child_qty_ok = bool((all_child_plus["filled_qty"] <= all_child_plus["requested_qty"] + 1e-8).all())
audit_rows.append({
    "check": "child_fills_do_not_exceed_requested",
    "value": child_qty_ok,
    "passed": child_qty_ok,
})

fill_rates_ok = bool(((all_parent_plus["fill_rate"] >= 0) & (all_parent_plus["fill_rate"] <= 1)).all())
audit_rows.append({
    "check": "parent_fill_rates_valid",
    "value": fill_rates_ok,
    "passed": fill_rates_ok,
})

costs_finite = bool(np.isfinite(all_parent_plus[["avg_realised_cost_bps", "avg_expected_cost_bps"]].to_numpy()).all())
audit_rows.append({
    "check": "parent_costs_finite",
    "value": costs_finite,
    "passed": costs_finite,
})

expected_policies = set(policies + ["optimised_split"])
actual_policies = set(all_parent_plus["policy"].unique())
policies_present = bool(expected_policies.issubset(actual_policies))
audit_rows.append({
    "check": "all_policies_present",
    "value": policies_present,
    "passed": policies_present,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

venues.to_csv(output_dir / "venue_metadata.csv", index=False)
market.to_csv(output_dir / "consolidated_market.csv", index=False)
venue_books.to_csv(output_dir / "venue_books.csv", index=False)
parents.to_csv(output_dir / "parent_orders.csv", index=False)
all_parent.to_csv(output_dir / "parent_results_base_policies.csv", index=False)
all_child.to_csv(output_dir / "child_results_base_policies.csv", index=False)
parent_opt.to_csv(output_dir / "parent_results_optimised_split.csv", index=False)
child_opt.to_csv(output_dir / "child_results_optimised_split.csv", index=False)
all_parent_plus.to_csv(output_dir / "parent_results_all_policies.csv", index=False)
all_child_plus.to_csv(output_dir / "child_results_all_policies.csv", index=False)
policy_summary.to_csv(output_dir / "policy_summary_base.csv", index=False)
venue_tca.to_csv(output_dir / "venue_tca_base.csv", index=False)
policy_summary_plus.to_csv(output_dir / "policy_summary_all.csv", index=False)
venue_tca_plus.to_csv(output_dir / "venue_tca_all.csv", index=False)
stress_performance.to_csv(output_dir / "stress_performance.csv", index=False)
cost_error_report.to_csv(output_dir / "cost_error_report.csv", index=False)
urgency_report.to_csv(output_dir / "urgency_report.csv", index=False)
audit_examples.to_csv(output_dir / "routing_audit_examples.csv")
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "smart_order_router_simulator_outputs",
    "schema_version": "smart_order_router_simulator_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "venues": venues.to_dict(orient="records"),
    "policies": policies + ["optimised_split"],
    "selected_policy": selected_policy,
    "policy_summary_all": policy_summary_plus.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook simulates smart order routing across synthetic venues.",
        "Venues differ by latency, fees, rebates, depth, fill intensity and adverse selection.",
        "Routing policies include primary-only, cheapest taker, fastest, liquidity-weighted, expected-cost, fill-probability-aware, dark-first and optimised split.",
        "Parent orders are split into child orders and evaluated using realised cost and fill rates.",
        "Venue-level TCA, stress performance, expected-vs-realised cost diagnostics and governance flags are reported.",
        "The data is synthetic and should be replaced with real venue quote, order and fill data before production use."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "policy_summary_all.csv", output_dir / "venue_tca_all.csv", output_dir / "governance_flags.csv", manifest_path

## 25. Practical checklist for smart order routers

Before trusting a router:

1. **Model venue fees and rebates explicitly.**
2. **Use venue-specific latency.**
3. **Separate marketable and passive logic.**
4. **Estimate fill probabilities by venue.**
5. **Estimate adverse selection by venue.**
6. **Run venue-level TCA.**
7. **Compare expected and realised costs.**
8. **Stress liquidity and spread regimes.**
9. **Check venue concentration.**
10. **Handle dark-pool minimum fills and fallback logic.**

## 26. Limitations

### Synthetic venue data

The notebook uses synthetic venue books and fills.

### Simplified routing logic

Real routers manage order acknowledgements, rejects, cancels, replaces, throttles, order protection, self-trade prevention, and routing priorities.

### No real queue position

Venue queue position is approximated through depth and fill intensity.

### Simplified dark pool

Dark midpoint fills are modelled probabilistically, without conditional order types or information leakage.

### No regulatory rules

The simulator does not implement Reg NMS, MiFID, best execution rules, or exchange-specific order types.

### No network model

Latency is event-based, not a real network-time model.

### No child-order persistence

Child order lifecycle is simplified.

## 27. Research frontier and extensions

Important extensions include:

1. real venue-specific fill models;
2. queue-position-aware routing;
3. reinforcement-learning routing;
4. dark-pool conditional order logic;
5. venue toxicity and adverse-selection models;
6. child-order lifecycle simulation;
7. order reject and cancel/replace simulation;
8. regulatory best-execution reporting;
9. self-trade prevention;
10. production router replay against historical drop-copy/fill data.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_11_execution_risk_controls_and_kill_switch**  
   Add hard risk controls to router decisions.

2. **06_12_l1_bid_ask_backtest_execution_model**  
   Integrate venue routing into a full backtest.

3. **06_13_production_reconciliation_and_breaks**  
   Reconcile intended route, acknowledged route, venue fills and broker records.

4. **Phase 7 execution capstone**  
   Combine impact, fill probability, market making and smart routing into a full execution research project.

## 29. Summary

This notebook implemented a smart order router simulator.

It showed how to:

1. define multi-venue metadata;
2. simulate consolidated and venue-specific market states;
3. generate parent orders;
4. split parents into child orders;
5. score venues by expected cost;
6. model marketable and passive routing;
7. compare routing policies;
8. simulate fills and realised costs;
9. compute venue-level TCA;
10. analyse stress-regime behaviour;
11. compare expected and realised cost;
12. optimise simple venue splits;
13. analyse routing by urgency;
14. produce audit examples;
15. create governance flags and audit checks;
16. save outputs and manifests.

The key computational takeaway:

> Smart routing is an optimisation problem over venue state, child size, fill probability and expected cost.

The key financial takeaway:

> The cheapest visible venue is not always the best venue once fill uncertainty, latency and adverse selection are included.

## 30. Further reading

- Harris, *Trading and Exchanges.*
- Kissell, *The Science of Algorithmic Trading and Portfolio Management.*
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Abergel et al., *Limit Order Books.*
- SEC and MiFID best-execution materials.
- Venue-level transaction cost analysis and smart order routing literature.